In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
pip install --upgrade pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 40.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [ ]:
import os, io, base64, subprocess, time, json
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
import cv2
from PIL import Image
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
llm = ChatOpenAI(model="gpt-4o-mini")
vision = ChatOpenAI(model="gpt-4o-mini", max_tokens=200)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
VIDEO_PATH = Path("/content/drive/MyDrive/Colab Notebooks/data/video/earnings_briefing.mp4")
WORK_DIR = Path("/content/drive/MyDrive/Colab Notebooks/data/day53_video")
WORK_DIR.mkdir(exist_ok=True)

In [ ]:
path = str(VIDEO_PATH)
cap = cv2.VideoCapture(path) # 경로에 있는 객체를 캡쳐해서 cap이라는 변수에 넣음

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS) # CAP_PROP_FPS는 fps를 가져오는 파라미터
print(fps)

24.0


In [ ]:
n_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
print(n_frames) # 영상의 총 프레임 수

717.0


In [ ]:
717 / 24

29.875

In [ ]:
w = cap.get(cv2.CAP_PROP_FRAME_WIDTH) # 영상 이미지의 크기(너비, 높이)
h = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
print(w, h)

720.0 1018.0


In [ ]:
def video_info(path):
  cap = cv2.VideoCapture(path)
  fps = cap.get(cv2.CAP_PROP_FPS)
  n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
  w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

  cap.release()
  return {'fps': round(fps,2), 'n_frames': n_frames, 'duration_sec': round(n_frames / fps if fps else 0, 2), 'width': w, 'height': h}

In [ ]:
def sample_frames(video_path, every_sec, out_dir=None): # every_sec은 매 n초 간격으로 샘플
  out_dir = Path(out_dir or WORK_DIR / 'frames')
  out_dir.mkdir(parents = True, exist_ok = True)
  info_ = video_info(video_path)
  cap = cv2.VideoCapture(video_path)
  items = []
  sec = 0.0
  i = 0

  while sec <= info_['duration_sec']:
    cap.set(cv2.CAP_PROP_POS_MSEC, sec * 1000) # POS 포지션, 현재 몇초에 있는지를 mili-sec 단위로
    ok, frame = cap.read()
    print(frame.shape)
    if not ok:
      break

    p = out_dir / f"frame_{i:03d}_{int(sec*1000):07d}ms.jpg" # 파일명
    cv2.imwrite(str(p), frame) # imwrite 자체가 경로에다 이미지를 저장함
    items.append({'path': str(p), 'second': round(sec,2)})
    sec += every_sec

    i += 1

  cap.release()
  return items

In [ ]:
frames = sample_frames(str(VIDEO_PATH), every_sec = 10)


(1018, 720, 3)
(1018, 720, 3)
(1018, 720, 3)


In [ ]:
# ffmpeg 라이브러리
# 영상에서 음성 파일을 추출하는 역할
# 공식 사이트에서..

In [ ]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [ ]:
def extract_audio(video_path, out_path): # 비디오 경로에서 음성을 분리해 out_path에 저장하는 함수
  if Path(out_path).exists():
    return out_path

  cmd = ['ffmpeg', '-y', '-i', video_path, '-vn', '-c:a', 'libmp3lame', '-b:a', '128k', out_path] # 경로 및 포맷 등

  r = subprocess.run(cmd, capture_output = True, text = True)
  if r.returncode != 0:
    raise RuntimeError(r.stderr[-300:])
  return out_path

In [ ]:
audio_path = extract_audio(str(VIDEO_PATH), str(WORK_DIR / 'audio.mp3'))

In [ ]:
# 오디오 -> 텍스트 변환

from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser
from langchain_core.document_loaders.blob_loaders import Blob

In [ ]:
AUDIO_DIR = Path('/content/drive/MyDrive/Colab Notebooks/data/audio')
audio_files = sorted(AUDIO_DIR.glob('*.mp3')) # a1.mp3, a2.mp3, ....등


In [ ]:
for f in audio_files:
  print(f"{f.name}: {f.stat().st_size} bytes")

meeting_hiring.mp3: 542880 bytes
meeting_product.mp3: 532320 bytes
meeting_revenue.mp3: 533760 bytes


In [ ]:
parser = OpenAIWhisperParser() # OpenAI의 음성 모델
sample = audio_files[0]
blob = Blob.from_path(str(sample)) # Blob 객체

In [ ]:
docs = list(parser.lazy_parse(blob))

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Transcribing part 1!


In [ ]:
docs[0].page_content

'인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'

In [ ]:
def transcribe_all(audio_paths):
  parser = OpenAIWhisperParser()
  out = []
  for p in audio_paths:
    blob = Blob.from_path(str(p))
    docs = list(parser.lazy_parse(blob))
    text = '\n'.join(d.page_content for d in docs)
    out.append({'source': Path(p).name, 'text': text})

  return out

In [ ]:
results = transcribe_all(audio_files)
print(results)

Transcribing part 1!
Transcribing part 1!
Transcribing part 1!
[{'source': 'meeting_hiring.mp3', 'text': '인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'}, {'source': 'meeting_product.mp3', 'text': '제품 출시 일정을 공유 드립니다. MODUFONE X는 3월 15일 정식 출시되며 가격은 129만원으로 책정되었습니다. MODUBOOK AIR는 14인치 OLED 디스플레이를 탑재한 초경량 노트북으로 4월 1일 출시 예정입니다. MODUPAD PRO는 M.SIME 칩셋을 탑재한 프로용 태블릿으로 가격은 149만원입니다. 마케팅 캠페인은 2월 말부터 시작됩니다.'}, {'source': 'meeting_revenue.mp3', 'text': '오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며 가장 큰 성장을 보였습니다. 영업이익은 24억 원으로 전년 동기 대비 35% 늘었습니다. 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.'}]


In [ ]:
# 어느 파일에서 왔는지, 추출된 텍스트의 원래 포맷은 오디오인지, 이미지인지 등등
  # 메타데이터를 잘 작성하는 것을 추천
def to_documents(transcripts):
  docs = []
  for r in transcripts:
    path = AUDIO_DIR / r['source']
    docs.append(Document(
        page_content = r['text'],
        metadata = {'source': r['source'], 'modality': 'audio'}
    ))
  return docs

In [ ]:
audio_docs = to_documents(results)

In [ ]:
audio_docs

[Document(metadata={'source': 'meeting_hiring.mp3', 'modality': 'audio'}, page_content='인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'),
 Document(metadata={'source': 'meeting_product.mp3', 'modality': 'audio'}, page_content='제품 출시 일정을 공유 드립니다. MODUFONE X는 3월 15일 정식 출시되며 가격은 129만원으로 책정되었습니다. MODUBOOK AIR는 14인치 OLED 디스플레이를 탑재한 초경량 노트북으로 4월 1일 출시 예정입니다. MODUPAD PRO는 M.SIME 칩셋을 탑재한 프로용 태블릿으로 가격은 149만원입니다. 마케팅 캠페인은 2월 말부터 시작됩니다.'),
 Document(metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio'}, page_content='오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며 가장 큰 성장을 보였습니다. 영업이익은 24억 원으로 전년 동기 대비 35% 늘었습니다. 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.')]

In [ ]:
# 인덱싱은 FAISS로

In [ ]:
from openai import OpenAI
oai_audio = OpenAI()

In [ ]:
def transcribe_with_segments(audio_path) -> dict:
  with open(audio_path, 'rb') as f:
    res = oai_audio.audio.transcriptions.create(
        model = 'whisper-1',
        file = f,
        response_format = 'verbose_json', # 언제 시작해서, 언제 끝나는지 등등의 포맷
        language = 'ko'
    )

    segs = [{'id' : s.id, 'start': s.start, 'end': s.end, 'text': s.text}
             for s in (res.segments or [])]

  return {'text': res.text , 'segments': segs}

In [ ]:
audio2 = transcribe_with_segments(str(audio_files[0]))

In [ ]:
audio2

{'text': '인사팀에서 채용 계획을 공유합니다. 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다. 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다. 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다. 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.',
 'segments': [{'id': 0,
   'start': 0.0,
   'end': 3.5,
   'text': ' 인사팀에서 채용 계획을 공유합니다.'},
  {'id': 1,
   'start': 3.5,
   'end': 11.300000190734863,
   'text': ' 올해 상반기에 AI 엔지니어 2명, 백엔드 개발자 15명, 데이터 분석가 10명을 채용할 예정입니다.'},
  {'id': 2,
   'start': 11.300000190734863,
   'end': 16.700000762939453,
   'text': ' 특히 AI 엔지니어는 멀티모달 ARG 경험자를 우대합니다.'},
  {'id': 3,
   'start': 16.700000762939453,
   'end': 22.65999984741211,
   'text': ' 신규 입사자 옴보딩은 4주 과정으로 진행되며 한국어와 영어 두 언어로 제공됩니다.'},
  {'id': 4,
   'start': 22.65999984741211,
   'end': 27.020000457763672,
   'text': ' 지원자는 회사 홈페이지의 채용 페이지를 통해 지원해 주시기 바랍니다.'}]}

In [ ]:
def segments_to_documents(audio_path) -> list:
    result = transcribe_with_segments(audio_path)
    docs = []
    for seg in result["segments"]:
        text = seg["text"].strip()
        if not text:
            continue
        doc = Document(
            page_content=text,
            metadata={
                "source": str(audio_path),
                "filename": Path(audio_path).name,
                "type": "audio",
                "id": seg["id"],
                "start": seg["start"],
                "end": seg["end"],
            }
        )

        docs.append(doc)

    return docs

In [ ]:
# 강사님 코드
def segments_to_documents(audio_path) -> list:
  tmp = transcribe_with_segments(audio_path)
  source = Path(audio_path).name
  docs = []
  for s in tmp['segments']:
    if not s['text'].strip():
      continue
    docs.append(Document(
        page_content = s['text'].strip(),
        metadata = {'source': source, 'modality': 'audio', 'start': s["start"], "end": s["end"],  "segment_id": s["id"]}
    ))

  return docs

In [ ]:
seg_docs = []
for f in audio_files:
  seg_docs.extend(segments_to_documents(str(f)))

In [ ]:
len(seg_docs)

21

In [ ]:
seg_vs = FAISS.from_documents(seg_docs, embeddings)
hits = seg_vs.similarity_search('매출', k=3)
print(hits)

[Document(id='1f20a96c-1d1a-48d4-b5ba-347e1ce01a66', metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio', 'start': 3.0799999237060547, 'end': 6.71999979019165, 'segment_id': 1}, page_content='2025년 4분기 매출은 188억 원으로'), Document(id='648a1645-7a88-4564-a933-ba1e5142b376', metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio', 'start': 17.079999923706055, 'end': 19.040000915527344, 'segment_id': 5}, page_content='영업이익은 24억 원으로'), Document(id='ecf6ae34-971f-4338-a06f-d54ec34656c9', metadata={'source': 'meeting_revenue.mp3', 'modality': 'audio', 'start': 9.920000076293945, 'end': 14.15999984741211, 'segment_id': 3}, page_content='특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며')]


In [ ]:
# 출력 형식
# outputparser -> json 형태의 응답 등등
# 또는 pydantic 이용

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class AudioCategory(BaseModel):
  # 주제, confidence, 이유
  category : Literal['revenue', 'product', 'hiring', 'other'] = Field(..., description = '주제 카테고리') # 리터럴은 카테고리를 강제, 4개 중 하나로..
  confidence : float = Field(..., ge=0, le=1, description = '확신도 0~1')
  reason : str = Field(...,description = '한국어 짧은 사유')

In [ ]:
classifier_llm = llm.with_structured_output(AudioCategory) # llm에 데이터 스키마 (Pydandic의 BaseModel) 추가했음 -> with_structured_output 적용함


In [ ]:
def classify_audi_doc(doc: Document) -> AudioCategory: # Document를 하나 받아서, 4개 중 하나의 카테고리를 알려주는 함수
    prompt = f"""
        오디오에서 추출된 텍스트를 다음 카테고리 중 하나로 분류하세요.
          - revenue
          - product
          - hiring
          - other

        텍스트:
        {doc.page_content}
    """

    result = classifier_llm.invoke([
        SystemMessage(content="당신은 오디오 내용을 분류하는 분류기입니다."),
        HumanMessage(content=prompt)
    ])

    return result

다시 영상 포맷으로 돌아와서..

In [ ]:
def transcribe_with_segments(audio_path) -> dict:
  with open(audio_path, 'rb') as f:
    res = oai_audio.audio.transcriptions.create(
        model = 'whisper-1',
        file = f,
        response_format = 'verbose_json', # 언제 시작해서, 언제 끝나는지 등등의 포맷
        language = 'ko'
    )

    segs = [{'id' : s.id, 'start': s.start, 'end': s.end, 'text': s.text}
             for s in (res.segments or [])]

  return {'text': res.text , 'segments': segs}

In [ ]:
audio_seg = transcribe_with_segments(audio_path)

In [ ]:
audio_seg

{'text': '오늘 분기 실적 회의를 시작하겠습니다. 2025년 4분기 매출은 188억 원으로 전 분기 대비 16% 증가했습니다. 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며 가장 큰 성장을 보였습니다. 영업이익은 24억 원으로 전년 동기 대비 35% 늘었습니다. 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.',
 'segments': [{'id': 0,
   'start': 0.0,
   'end': 3.0799999237060547,
   'text': ' 오늘 분기 실적 회의를 시작하겠습니다.'},
  {'id': 1,
   'start': 3.0799999237060547,
   'end': 6.679999828338623,
   'text': ' 2025년 4분기 매출은 188억 원으로'},
  {'id': 2,
   'start': 6.679999828338623,
   'end': 9.920000076293945,
   'text': ' 전 분기 대비 16% 증가했습니다.'},
  {'id': 3,
   'start': 9.920000076293945,
   'end': 14.15999984741211,
   'text': ' 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며'},
  {'id': 4,
   'start': 14.15999984741211,
   'end': 17.079999923706055,
   'text': ' 가장 큰 성장을 보였습니다.'},
  {'id': 5,
   'start': 17.079999923706055,
   'end': 19.040000915527344,
   'text': ' 영업이익은 24억 원으로'},
  {'id': 6,
   'start': 19.040000915527344,
   'end': 22.920000076293945,
   'text': ' 전년 동기 대비 35% 늘었습니다.'},
  {'id': 7,
   'start': 22.920000076293945,


In [ ]:
def _b64(path):
  return base64.b64encode(Path(path).read_bytes()).decode()

def caption_frames(frames:list, prompt:str='이 프레임에 무엇이 보이는지 한 문장으로 써주세요') -> list:
  out = []
  for fr in frames:
    b = _b64(fr['path'])
    msg = HumanMessage(content = [
        {'type': 'text', 'text': prompt},
        {'type': 'image_url', 'image_url' : {"url": f"data:image/jpeg;base64,{b}"}}

    ])
    cap = vision.invoke([msg]).content
    out.append({**fr, 'caption': cap})

  return out

In [ ]:
captioned = caption_frames(frames)

## Video RAG

In [ ]:
def build_video_documents(frames, segments, source): # frame은 이미지에서, segment는 오디오에서
  docs = [] # 빈 리스트 만들고
  for fr in frames:
    docs.append(Document(
        page_content = fr['caption'],
        metadata = {'source': source, 'modality': 'frame', 'second': fr['second'], 'frame_path': fr['path']}
    ))

  for seg in segments:
    docs.append(Document(
        page_content = seg['text'].strip(),
        metadata = {'source': source, 'modality': 'audio', 'start': round(seg['start'],2), 'end': round(seg['end'], 2)}
    ))

  return docs



In [ ]:
video_docs = build_video_documents(captioned, audio_seg['segments'], source = VIDEO_PATH.name)


In [ ]:
len(video_docs)

11

In [ ]:
for d in video_docs:
  if d.metadata['modality'] == 'frame':
    print(f" [frame @ {d.metadata['second']}s {d.page_content[:50]}]")

  else:
    print(f" [audio {d.metadata['start']} - {d.metadata['end']}] {d.page_content[:50]}")

 [frame @ 0.0s 이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 매출이 188억 원, 영업]
 [frame @ 10.0s 2025년 4분기 실적 보고서로, 총 매출 188억 원과 영업이익 24억 원을 기록하며 전]
 [frame @ 20.0s 이 프레임에는 2024년 1월부터 2025년 6월까지의 월별 매출 추이와 시장 평균을 비교]
 [audio 0.0 - 3.08] 오늘 분기 실적 회의를 시작하겠습니다.
 [audio 3.08 - 6.68] 2025년 4분기 매출은 188억 원으로
 [audio 6.68 - 9.92] 전 분기 대비 16% 증가했습니다.
 [audio 9.92 - 14.16] 특히 AI 솔루션 사업부가 98억 원의 매출을 기록하며
 [audio 14.16 - 17.08] 가장 큰 성장을 보였습니다.
 [audio 17.08 - 19.04] 영업이익은 24억 원으로
 [audio 19.04 - 22.92] 전년 동기 대비 35% 늘었습니다.
 [audio 22.92 - 26.6] 다음 분기는 동남아시아 시장 진출 본격화할 계획입니다.


In [ ]:
# video_docs -> vectorstore indexing 하는것
# 만약 매출 188억 검색하면 어떤게 뜨는지?

video_vs = FAISS.from_documents(video_docs, embeddings)

hits = video_vs.similarity_search("매출 188억", k=3)

for h in hits:
    print(h.page_content)
    print(h.metadata)
    print("------------")

2025년 4분기 매출은 188억 원으로
{'source': 'earnings_briefing.mp4', 'modality': 'audio', 'start': 3.08, 'end': 6.68}
------------
2025년 4분기 실적 보고서로, 총 매출 188억 원과 영업이익 24억 원을 기록하며 전년 동기 대비 매출이 35% 증가한 내용을 담고 있습니다.
{'source': 'earnings_briefing.mp4', 'modality': 'frame', 'second': 10.0, 'frame_path': '/content/drive/MyDrive/Colab Notebooks/data/day53_video/frames/frame_001_0010000ms.jpg'}
------------
이 문서는 Modu Tech의 2025년 4분기 실적 보고서로, 매출이 188억 원, 영업이익이 24억 원에 달하며, 전년 동기 대비 각각 35%와 42% 성장했음을 요약하고 있습니다.
{'source': 'earnings_briefing.mp4', 'modality': 'frame', 'second': 0.0, 'frame_path': '/content/drive/MyDrive/Colab Notebooks/data/day53_video/frames/frame_000_0000000ms.jpg'}
------------


In [ ]:
video_vs.index.ntotal

11

In [ ]:
def video_rag_visual(vs, query, k=4):
  cands = vs.similarity_search(query, k=k)
  text_blocks = []
  image_paths = []

  for d in cands:
    m = d.metadata
    if m['modality'] == 'frame':
      image_paths.append((m['second'], m['frame_path'], d.page_content))

    else:
      text_blocks.append(f"[audio {m['start']}--{m['end']}s] {d.page_content}")

  content = [{'type' : 'text',
              'text' : f"비디오에서 발췌한 프레임 이미지와 음성 transcribe을 참고하여 한국어로 답하세요.\n\n질문: {query}"
              }]
  if text_blocks:
    content.append({'type': 'text', 'text' : '[음성발췌]\n' + '\n'.join(text_blocks)})

  for sec, path, cap in image_paths:
    content.append({'type': 'text', 'text': f"\n[frame @ {sec}s] (caption: {cap}) "})
    content.append({'type': 'image_url', 'image_url': {'url': f"data:image/jpeg:base64,{_b64(path)}"}})


  msg = HumanMessage(content = content)
  return {'answer': vision.invoke([msg]).content, 'n_frames': len(image_paths), 'n_audio': len(text_blocks)}

In [ ]:
r = video_rag_visual(video_vs, '비디오 첫 장면에 어떤 차트가 보이나요?', k = 3)

In [ ]:
r

{'answer': '비디오 첫 장면에는 2024년 1월부터 2025년 6월까지의 월별 매출 추이와 시장 평균을 비교한 그래프가 보입니다. 그래프에는 월별 매출 추이가 표시되어 있으며, 주요 KPI가 포함되어 있습니다.',
 'n_frames': 1,
 'n_audio': 2}